# ML5_Decision trees

In [1]:
!pip install --upgrade scikit-learn

In [2]:
import numpy as np
import pandas as pd
from scipy.stats import randint, uniform

from category_encoders import CountEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.metrics import roc_auc_score

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

In [3]:
import lightgbm as lgb
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

In [4]:
from my_utils import time_ordered_split_3_new
from my_utils import update_model_results

## 1. Download data

In [5]:
df = pd.read_csv("../datasets/train.csv")

In [6]:
y = df["IsBadBuy"].copy()
df = df.drop(columns="IsBadBuy")
df = df.drop("VehYear", axis=1)
X_copy, y_copy = df.copy(), y.copy()

numeric_cols = df.select_dtypes(include=["number"]).columns
df[numeric_cols] = df[numeric_cols].fillna(0.0)

non_numeric_cols = df.select_dtypes(exclude=["number"]).columns
df[non_numeric_cols] = df[non_numeric_cols].fillna("unknown")

X_train, X_valid, X_test, y_train, y_valid, y_test = time_ordered_split_3_new(
    df,
    y,
    date_field="PurchDate",
    train_size=1 / 3,
    valid_size=1 / 3,
    test_size=1 / 3,
)


print(f"Train dates: {X_train['PurchDate'].min()} to {X_train['PurchDate'].max()}")
print(f"Valid dates: {X_valid['PurchDate'].min()} to {X_valid['PurchDate'].max()}")
print(f"Test dates: {X_test['PurchDate'].min()} to {X_test['PurchDate'].max()}")

X_train = X_train.drop(columns="PurchDate")
X_valid = X_valid.drop(columns="PurchDate")
X_test = X_test.drop(columns="PurchDate")

Train dates: 2009-01-05 00:00:00 to 2009-09-01 00:00:00
Valid dates: 2009-09-02 00:00:00 to 2010-04-30 00:00:00
Test dates: 2010-05-03 00:00:00 to 2010-12-30 00:00:00


In [7]:
print(len(X_train))
print(len(X_valid))
print(len(X_test))

23059
24104
25820


In [8]:
ohe_cols = [
    "Auction",
    "Color",
    "Transmission",
    "WheelType",
    "Nationality",
    "Size",
    "TopThreeAmericanName",
    "PRIMEUNIT",
    "AUCGUART",
]
ce_cols = ["Make", "Model", "Trim", "SubModel", "VNST"]
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()

preprocessor = ColumnTransformer(
    [
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ohe_cols,
        ),
        (
            "count",
            Pipeline(
                [
                    ("encoder", CountEncoder(handle_unknown=0)),
                    ("scaler", StandardScaler()),
                ]
            ),
            ce_cols,
        ),
        ("numeric", StandardScaler(), numeric_cols),
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_valid_processed = preprocessor.transform(X_valid)
X_test_processed = preprocessor.transform(X_test)

ohe_feature_names = preprocessor.named_transformers_["onehot"].get_feature_names_out(
    ohe_cols
)
ce_feature_names = ce_cols
numeric_names = numeric_cols

all_feature_names = list(ohe_feature_names) + ce_feature_names + numeric_names

X_train_processed = pd.DataFrame(
    X_train_processed, columns=all_feature_names, index=X_train.index
)
X_valid_processed = pd.DataFrame(
    X_valid_processed, columns=all_feature_names, index=X_valid.index
)
X_test_processed = pd.DataFrame(
    X_test_processed, columns=all_feature_names, index=X_test.index
)

In [9]:
results = pd.DataFrame()


## 2. Model Custom

### 2.0. Node

In [10]:
class Node:
    """
    A node in a decision tree.

    Represents either an internal decision node that splits data based on a
    feature and threshold, or a leaf node that stores a predicted value.

    Attributes
    ----------
    feature_index : int or None
        Index of the feature used for splitting at this node.
        None for leaf nodes.
    threshold : float or None
        Threshold value for the feature split. Samples with feature value
        less than or equal to threshold go to the left child, others to
        the right child. None for leaf nodes.
    left : Node or None
        Left child node containing samples that satisfied the split condition.
        None for leaf nodes.
    right : Node or None
        Right child node containing samples that did not satisfy the split
        condition. None for leaf nodes.
    value : float or None
        Predicted value stored at the leaf node (e.g., class label or
        regression target). If not None, this node is a leaf node.
        None for internal decision nodes.

    Methods
    -------
    is_leaf_node()
        Returns True if this node is a leaf node (has a predicted value),
        False otherwise.
    """

    def __init__(
        self, feature_index=None, threshold=None, left=None, right=None, value=None
    ):
        self.feature_index = feature_index
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

    def is_leaf_node(self):
        """Return True if the node is a leaf node, False otherwise."""
        return self.value is not None

### 2.1. Decision Tree Classifier

In [11]:
class DecisionTreeClassifier_custom:
    """
    A custom decision tree classifier built from scratch using NumPy.

    Implements the CART (Classification and Regression Trees) algorithm
    with Gini impurity as the splitting criterion. Supports configurable
    maximum depth, random feature subset selection at each split, and
    returns class probabilities at leaf nodes.

    Parameters
    ----------
    max_depth : int, default=7
        Maximum depth of the decision tree. The root node has depth 0.
        Prevents overfitting by limiting tree complexity.
    max_features : int, str, or None, default=None
        Number of features to consider when looking for the best split.
        - If None, all features are considered.
        - If "sqrt", max_features = int(sqrt(n_features)).
        - If int, the specified number of features is randomly selected
          at each split.
    random_state : int or None, default=None
        Seed for the random number generator used in feature subsampling.
        Pass an int for reproducible output across multiple function calls.

    Attributes
    ----------
    root : Node or None
        The root node of the fitted decision tree. None if the model has
        not been fitted.
    n_classes_ : int or None
        Number of unique classes in the training data. Set during fitting.
        None if the model has not been fitted.

    """

    def __init__(self, max_depth=7, max_features=None, random_state=None):
        self.max_depth = max_depth
        self.max_features = max_features
        self.random_state = random_state
        self.root = None
        self.n_classes_ = None

    def fit(self, X, y):
        """
        Build the decision tree from training data.

        Parameters
        ----------
        X : array-like of shape (n_samples, n_features)
            Training feature matrix. Accepts NumPy arrays or Pandas
            DataFrames/Series, which are converted to NumPy arrays
            internally.
        y : array-like of shape (n_samples,)
            Target vector (class labels) as integers. Accepts NumPy
            arrays or Pandas Series, converted to NumPy arrays
            internally.

        Returns
        -------
        None
            Fits the model in-place by setting `self.root` and
            `self.n_classes_`.
        """
        if isinstance(X, pd.DataFrame):
            X = X.values
        if isinstance(y, pd.Series):
            y = y.values

        self.n_classes_ = len(np.unique(y))
        self.root = self._build_tree(X, y, depth=0)

    def _build_tree(self, X, y, depth):
        """
        Recursively build the decision tree.

        Parameters
        ----------
        X : np.ndarray of shape (n_samples, n_features)
            Feature matrix for the current subset of samples.
        y : np.ndarray of shape (n_samples,)
            Target labels for the current subset of samples.
        depth : int
            Current depth of the tree (root is 0).

        Returns
        -------
        Node
            The constructed decision tree node (internal or leaf).
            Reshapes 1D or 3D inputs to 2D for consistency.
            Returns a leaf node if any stopping criterion is met:
            - depth >= max_depth
            - only one class remains
            - fewer than 2 samples
        """
        if isinstance(X, pd.DataFrame):
            X = X.values
        if isinstance(X, pd.Series):
            X = X.values.reshape(1, -1)

        if isinstance(X, np.ndarray):
            if X.ndim == 1:
                X = X.reshape(1, -1)
            elif X.ndim == 3:
                X = X.reshape(-1, X.shape[-1])

        n_samples, n_features = X.shape
        n_labels = len(np.unique(y))

        if depth >= self.max_depth or n_labels == 1 or n_samples < 2:
            leaf_value = self._calculate_leaf_value(y)
            return Node(value=leaf_value)

        best_feat, best_thresh = self._best_split(X, y, n_features)

        if best_feat is None:
            leaf_value = self._calculate_leaf_value(y)
            return Node(value=leaf_value)

        left_idxs, right_idxs = self._split(X[:, best_feat], best_thresh)
        left_child = self._build_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right_child = self._build_tree(X[right_idxs, :], y[right_idxs], depth + 1)

        return Node(
            feature_index=best_feat,
            threshold=best_thresh,
            left=left_child,
            right=right_child,
        )

    def _best_split(self, X, y, n_features):
        """
        Find the best feature and threshold to split on, maximizing
        information gain (Gini impurity reduction).

        Randomly selects a subset of features according to
        `self.max_features` before evaluating candidate splits.

        Parameters
        ----------
        X : np.ndarray of shape (n_samples, n_features)
            Feature matrix.
        y : np.ndarray of shape (n_samples,)
            Target labels.
        n_features : int
            Total number of features in X.

        Returns
        -------
        split_idx : int or None
            Index of the best feature to split on, or None if no valid
            split exists.
        split_thresh : float or None
            Threshold value for the best split, or None if no valid
            split exists.
        """
        best_gain = -1
        split_idx, split_thresh = None, None
        parent_gini = self._gini(y)

        if self.max_features == "sqrt":
            max_feat = int(np.sqrt(n_features))
        elif isinstance(self.max_features, int):
            max_feat = self.max_features
        else:
            max_feat = n_features

        if self.random_state is not None:
            np.random.seed(self.random_state)

        feat_indices = np.random.choice(n_features, max_feat, replace=False)

        for feat_idx in feat_indices:
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)

            for thresh in thresholds:
                gain = self._information_gain(y, X_column, thresh, parent_gini)
                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat_idx
                    split_thresh = thresh

        return split_idx, split_thresh

    def _information_gain(self, y, X_column, split_thresh, parent_gini):
        """
        Compute information gain (Gini impurity reduction) for a split.

        Parameters
        ----------
        y : np.ndarray of shape (n_samples,)
            Target labels.
        X_column : np.ndarray of shape (n_samples,)
            Values of the candidate feature for all samples.
        split_thresh : float
            Threshold value to split on.
        parent_gini : float
            Gini impurity of the parent node before splitting.

        Returns
        -------
        float
            Information gain. Returns 0 if either child would be empty
            or the split indices are invalid.
        """
        left_idxs, right_idxs = self._split(X_column, split_thresh)

        if left_idxs is None or right_idxs is None:
            return 0

        if len(left_idxs) == 0 or len(right_idxs) == 0:
            return 0

        n = len(y)
        n_l, n_r = len(left_idxs), len(right_idxs)
        gini_l, gini_r = self._gini(y[left_idxs]), self._gini(y[right_idxs])

        child_gini = (n_l / n) * gini_l + (n_r / n) * gini_r
        ig = parent_gini - child_gini
        return ig

    def _gini(self, y):
        """
        Calculate Gini impurity for a set of labels.

        Gini impurity = 1 - sum(p_i^2), where p_i is the proportion of
        class i in the node. A value of 0 indicates perfect purity.

        Parameters
        ----------
        y : np.ndarray of shape (n_samples,)
            Array of integer class labels.

        Returns
        -------
        float
            Gini impurity value between 0 and 1 - (1 / n_classes_).
        """
        counts = np.bincount(y)
        probabilities = counts / len(y)
        return 1.0 - np.sum(probabilities**2)

    def _split(self, X_column, split_thresh):
        """
        Partition samples into left and right child indices based on
        a feature column and threshold.

        Left child: samples with X_column <= split_thresh.
        Right child: samples with X_column > split_thresh.

        Parameters
        ----------
        X_column : np.ndarray of shape (n_samples,)
            Feature values for a single feature across all samples.
        split_thresh : float
            Threshold value for the split.

        Returns
        -------
        left_idxs : np.ndarray or None
            Indices of samples assigned to the left child, or None if
            either child would be empty.
        right_idxs : np.ndarray or None
            Indices of samples assigned to the right child, or None if
            either child would be empty.
        """
        left_idxs = np.argwhere(X_column <= split_thresh).flatten()
        right_idxs = np.argwhere(X_column > split_thresh).flatten()

        if len(left_idxs) == 0 or len(right_idxs) == 0:
            return None, None

        return left_idxs, right_idxs

    def _calculate_leaf_value(self, y):
        """
        Compute the class probability distribution for a leaf node.

        Parameters
        ----------
        y : np.ndarray of shape (n_samples,)
            Target labels for samples reaching the leaf node.

        Returns
        -------
        np.ndarray of shape (n_classes_,)
            Normalized class probabilities. If `y` is empty, returns
            a uniform distribution over all classes.
        """
        if len(y) == 0:
            return np.ones(self.n_classes_) / self.n_classes_

        counts = np.bincount(y, minlength=self.n_classes_)
        probabilities = counts / len(y)
        return probabilities

    def predict_proba(self, X):
        """
        Predict class probabilities for samples in X.

        Parameters
        ----------
        X : array-like of shape (n_samples, n_features)
            Feature matrix. Accepts NumPy arrays or Pandas DataFrames.

        Returns
        -------
        np.ndarray of shape (n_samples, n_classes_)
            Predicted class probability vectors for each sample.
        """
        if isinstance(X, pd.DataFrame):
            X = X.values
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def predict(self, X):
        """
        Predict class labels for samples in X.

        Parameters
        ----------
        X : array-like of shape (n_samples, n_features)
            Feature matrix. Accepts NumPy arrays or Pandas DataFrames.

        Returns
        -------
        np.ndarray of shape (n_samples,)
            Predicted class labels (the class with the highest
            predicted probability).
        """
        proba = self.predict_proba(X)
        return np.argmax(proba, axis=1)

    def _traverse_tree(self, x, node):
        """
        Recursively traverse the tree to find the leaf node for a
        single sample, returning the node's probability distribution.

        Parameters
        ----------
        x : np.ndarray of shape (n_features,)
            Feature vector of a single sample.
        node : Node or None
            Current node in the decision tree. If None or a leaf node,
            the traversal stops.

        Returns
        -------
        np.ndarray of shape (n_classes_,)
            Class probability distribution. Returns a uniform
            distribution if `node` is None.
        """
        if node is None:
            return np.ones(self.n_classes_) / self.n_classes_

        if node.is_leaf_node():
            return node.value

        if x[node.feature_index] <= node.threshold:
            return self._traverse_tree(x, node.left)

        return self._traverse_tree(x, node.right)

    def gini(self, y):
        """
        Public interface for computing Gini impurity.

        Parameters
        ----------
        y : np.ndarray of shape (n_samples,)
            Array of integer class labels.

        Returns
        -------
        float
            Gini impurity value between 0 and 1 - (1 / n_classes_).
        """
        return self._gini(y)

### 2.2. Decision Tree Regressor

In [12]:
class DecisionTreeRegressor_custom(DecisionTreeClassifier_custom):
    """
    A custom decision tree regressor built on top of DecisionTreeClassifier_custom.

    Inherits the tree-building structure from the classifier but overrides
    the splitting criterion to use variance reduction (MSE reduction) instead
    of Gini impurity. Leaf nodes store the mean target value of their samples.
    This implements the CART regression algorithm.

    Parameters
    ----------
    max_depth : int, default=7
        Maximum depth of the decision tree. The root node has depth 0.
        Prevents overfitting by limiting tree complexity.
    max_features : int, str, or None, default=None
        Number of features to consider when looking for the best split.
        - If None, all features are considered.
        - If "sqrt", max_features = int(sqrt(n_features)).
        - If int, the specified number of features is randomly selected
          at each split.
    random_state : int or None, default=None
        Seed for the random number generator used in feature subsampling.
        Pass an int for reproducible output across multiple function calls.

    Attributes
    ----------
    root : Node or None
        The root node of the fitted decision tree. None if the model has
        not been fitted.
    n_classes_ : None
        Always None for the regressor (set during fitting for API
        consistency with the parent class). Not used for regression.

    Notes
    -----
    This class overrides the following methods from DecisionTreeClassifier_custom:
    - ``_best_split``: uses variance reduction instead of Gini gain.
    - ``_calculate_leaf_value``: returns the mean of y instead of class
      probabilities.
    - ``_traverse_tree``: returns a scalar prediction (leaf mean) instead
      of a probability array.
    - ``predict``: no argmax step; predictions are continuous values.
    - ``predict_proba``: explicitly disabled (raises NotImplementedError).

    """

    def __init__(self, max_depth=7, max_features=None, random_state=None):
        super().__init__(
            max_depth=max_depth, max_features=max_features, random_state=random_state
        )

    def fit(self, X, y):
        """
        Build the decision tree regressor from training data.

        Sets `n_classes_` to None (not applicable for regression) and
        constructs the tree by recursively partitioning the feature space
        using variance reduction as the splitting criterion.

        Parameters
        ----------
        X : array-like of shape (n_samples, n_features)
            Training feature matrix. Accepts NumPy arrays or Pandas
            DataFrames, which are converted to NumPy arrays internally.
        y : array-like of shape (n_samples,)
            Target vector (continuous values). Accepts NumPy arrays
            or Pandas Series, converted to NumPy arrays internally.

        Returns
        -------
        None
            Fits the model in-place by setting `self.root`.
        """
        self.n_classes_ = None
        if isinstance(X, pd.DataFrame):
            X = X.values
        if isinstance(y, pd.Series):
            y = y.values
        self.root = self._build_tree(X, y, depth=0)

    def _best_split(self, X, y, n_features):
        """
        Find the best feature and threshold to split on, maximizing
        variance reduction.

        Overrides the parent's Gini-based method. Randomly selects a
        subset of features according to `self.max_features` before
        evaluating candidate splits.

        Parameters
        ----------
        X : np.ndarray of shape (n_samples, n_features)
            Feature matrix.
        y : np.ndarray of shape (n_samples,)
            Continuous target values.
        n_features : int
            Total number of features in X.

        Returns
        -------
        split_idx : int or None
            Index of the best feature to split on, or None if no valid
            split exists.
        split_thresh : float or None
            Threshold value for the best split, or None if no valid
            split exists.
        """
        best_gain = -1
        split_idx, split_thresh = None, None

        parent_var = self._variance(y)

        if self.max_features == "sqrt":
            max_feat = int(np.sqrt(n_features))
        elif isinstance(self.max_features, int):
            max_feat = self.max_features
        else:
            max_feat = n_features

        if self.random_state is not None:
            np.random.seed(self.random_state)

        feat_indices = np.random.choice(n_features, max_feat, replace=False)

        for feat_idx in feat_indices:
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)

            for thresh in thresholds:
                gain = self._variance_reduction(y, X_column, thresh, parent_var)
                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat_idx
                    split_thresh = thresh

        return split_idx, split_thresh

    def _variance_reduction(self, y, X_column, split_thresh, parent_var):
        """
        Compute variance reduction (analogous to information gain) for
        a candidate split.

        Variance reduction = parent variance - weighted average of
        child variances. Higher values indicate a better split.

        Parameters
        ----------
        y : np.ndarray of shape (n_samples,)
            Continuous target values.
        X_column : np.ndarray of shape (n_samples,)
            Values of the candidate feature for all samples.
        split_thresh : float
            Threshold value to split on.
        parent_var : float
            Variance of the target values in the parent node.

        Returns
        -------
        float
            Variance reduction. Returns 0 if either child would be
            empty or the split indices are invalid.
        """
        left_idxs, right_idxs = self._split(X_column, split_thresh)

        if left_idxs is None or right_idxs is None:
            return 0

        if len(left_idxs) == 0 or len(right_idxs) == 0:
            return 0

        n = len(y)
        n_l, n_r = len(left_idxs), len(right_idxs)
        var_l, var_r = self._variance(y[left_idxs]), self._variance(y[right_idxs])

        child_var = (n_l / n) * var_l + (n_r / n) * var_r
        reduction = parent_var - child_var
        return reduction

    def _variance(self, y):
        """
        Calculate the variance of a set of continuous target values.

        Parameters
        ----------
        y : np.ndarray of shape (n_samples,)
            Array of continuous target values.

        Returns
        -------
        float
            Variance of y. Returns 0 for empty arrays.
        """
        if len(y) == 0:
            return 0
        return np.var(y)

    def _calculate_leaf_value(self, y):
        """
        Compute the prediction value for a leaf node (mean of y).

        Overrides the parent's probability distribution method.

        Parameters
        ----------
        y : np.ndarray of shape (n_samples,)
            Continuous target values for samples reaching the leaf node.

        Returns
        -------
        float
            Mean of y. Returns 0.0 if `y` is empty.
        """
        if len(y) == 0:
            return 0.0
        return np.mean(y)

    def predict_proba(self, X):
        """
        Not supported for regression.

        Raises
        ------
        NotImplementedError
            Always raised, as probability predictions are not
            applicable to regression tasks.
        """
        raise NotImplementedError("Regressors do not support predict_proba.")

    def predict(self, X):
        """
        Predict continuous target values for samples in X.

        Traverses the tree for each sample and returns the leaf node's
        mean value (scalar prediction).

        Parameters
        ----------
        X : array-like of shape (n_samples, n_features)
            Feature matrix. Accepts NumPy arrays or Pandas DataFrames.

        Returns
        -------
        np.ndarray of shape (n_samples,)
            Predicted continuous target values. Returns 0.0 for
            samples that reach a None node.
        """
        if isinstance(X, pd.DataFrame):
            X = X.values
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _traverse_tree(self, x, node):
        """
        Recursively traverse the tree for a single sample and return
        the leaf node's predicted value.

        Overrides the parent's method which returns a probability array.
        For regression, returns a scalar float.

        Parameters
        ----------
        x : np.ndarray of shape (n_features,)
            Feature vector of a single sample.
        node : Node or None
            Current node in the decision tree. If None, returns 0.0.
            If a leaf node, returns the stored mean value.

        Returns
        -------
        float
            Predicted value for the sample. Returns 0.0 if `node`
            is None.
        """
        if node is None:
            return 0.0

        if node.is_leaf_node():
            return node.value

        if x[node.feature_index] <= node.threshold:
            return self._traverse_tree(x, node.left)

        return self._traverse_tree(x, node.right)

## 3. Sklearn's DecisionTreeClassifier

In [13]:
model_custom = DecisionTreeClassifier_custom(max_depth=7)
model_custom.fit(X_train_processed, y_train)
proba_custom = model_custom.predict_proba(X_valid_processed)
gini_custom = 2 * roc_auc_score(y_valid, proba_custom[:, 1]) - 1
results = update_model_results(results, "DecisionTreeClassifier_custom", "valid", gini_custom)
gini_custom

0.42042814778374127

In [14]:
model = DecisionTreeClassifier(max_depth=7)
model.fit(X_train_processed, y_train)
proba = model.predict_proba(X_valid_processed)[:, 1]
gini = 2 * roc_auc_score(y_valid, proba) - 1
results = update_model_results(results, "DecisionTreeClassifier", "valid", gini)
gini

0.4206301616396151

### Validation Results
| Implementation | Gini | Speed |
|---------------|------|-------|
| Custom DT | 0.3707 | Slow |
| Sklearn DT | 0.3744 | Fast |

### Why sklearn wins:
1. **Cython** (not pure Python) → orders of magnitude faster loops.
2. **Smarter split search** (midpoints, not all unique values).
3. **Built-in pruning** (`min_samples_split`, etc.) → less overfitting.
4. **Better vectorization** via NumPy.
5. **Decades of production-grade development.**

## 4. RandomForestClassifier

In [24]:
class RandomForestClassifier_custom:
    """
    A custom random forest classifier built on top of DecisionTreeClassifier_custom.

    Implements an ensemble of decision trees trained on bootstrap samples
    (bagging) with random feature subsampling at each split. Predictions are
    made by averaging class probabilities across all trees and selecting the
    class with the highest mean probability (soft voting).

    This is an analogue of scikit-learn's RandomForestClassifier, built
    using the custom DecisionTreeClassifier_custom as the base estimator.

    Parameters
    ----------
    n_estimators : int, default=100
        Number of decision trees in the forest. More trees typically
        improve generalization but increase training and prediction time
        linearly.
    max_depth : int, default=7
        Maximum depth of each decision tree. Passed to each base estimator
        to control overfitting.
    max_features : int, str, or None, default="sqrt"
        Number of features to consider when looking for the best split in
        each tree.
        - If None, all features are considered.
        - If "sqrt", max_features = int(sqrt(n_features)).
        - If int, the specified number of features is randomly selected
          at each split.
    random_state : int or None, default=None
        Seed for the random number generator. If provided, each tree
        receives a different seed (random_state + i), ensuring diversity
        while maintaining reproducibility across runs.

    Attributes
    ----------
    trees : list of DecisionTreeClassifier_custom
        Fitted base decision tree estimators, one per n_estimators.
    n_classes_ : int or None
        Number of unique classes in the training data. Set during fitting.
        None if the model has not been fitted.

    Notes
    -----
    The training process for each tree involves:
    1. Bootstrap sampling: drawing `n_samples` samples with replacement
       from the original dataset.
    2. Training a DecisionTreeClassifier_custom on the bootstrap sample
       with feature subsampling at each node.

    Predictions use averaging:
    - ``predict_proba`` returns the mean probability vector across all trees.
    - ``predict`` returns the class with the highest mean probability.

    The `random_state` for individual trees is incremented (``random_state + i``)
    to ensure each tree uses a different sequence of random feature selections
    while keeping the overall process reproducible. If `random_state` is None,
    trees use independent random seeds.
    """
    def __init__(self, n_estimators=100, max_depth=7, max_features='sqrt', random_state=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.max_features = max_features
        self.random_state = random_state
        self.trees = []
        self.n_classes_ = None

    def fit(self, X, y):
        if isinstance(X, pd.DataFrame):
          X = X.values
        if isinstance(y, pd.Series):
          y = y.values

        if self.random_state is not None:
            np.random.seed(self.random_state)

        self.trees = []
        self.n_classes_ = len(np.unique(y))
        n_samples = X.shape[0]

        for i in range(self.n_estimators):
            indices = np.random.choice(n_samples, n_samples, replace=True)
            X_bootstrap = X[indices]
            y_bootstrap = y[indices]

            tree = DecisionTreeClassifier(max_depth=self.max_depth, max_features=self.max_features, random_state=self.random_state + i if self.random_state else None)
            tree.fit(X_bootstrap, y_bootstrap)
            self.trees.append(tree)

    def predict_proba(self, X):
      if isinstance(X, pd.DataFrame):
        X = X.values
      tree_probs = np.array([tree.predict_proba(X) for tree in self.trees])
      return np.mean(tree_probs, axis=0)

    def predict(self, X):
        proba = self.predict_proba(X)
        return np.argmax(proba, axis=1)

In [25]:
rf = RandomForestClassifier_custom(
    n_estimators=50, max_depth=7, max_features="sqrt", random_state=21
)
rf.fit(X_train_processed, y_train)
y_pred_proba = rf.predict_proba(X_valid_processed)[:, 1]
gini = 2 * roc_auc_score(y_valid, y_pred_proba) - 1
results = update_model_results(results, "RandomForestClassifier_custom", "valid", gini)
gini

0.4698758638000595

In [20]:
y_train.shape

(23059,)

## 5. Gradient Boosted Decision Trees — GBDT

In [26]:
class GBDTClassifier_custom:
    """
    Gradient Boosting Decision Tree classifier for binary classification.

    Uses logistic loss (negative log-likelihood) with decision tree regressors
    as base learners. Predictions are updated additively in the log-odds space.

    Parameters
    ----------
    max_depth : int, default=3
        Maximum depth of individual decision trees.
    number_of_trees : int, default=100
        Number of boosting iterations (trees).
    max_features : int or None, default=None
        Maximum number of features considered per split. If None, all features are used.
    learning_rate : float, default=0.1
        Shrinkage factor applied to each tree's contribution.
    random_state : int or None, default=None
        Seed for reproducibility. If provided, each tree receives a derived seed.

    Attributes
    ----------
    trees : list of DecisionTreeRegressor
        Fitted base learners.
    initial_prediction : float
        Log-odds of the positive class in the training data.
    classes_ : ndarray
        Unique class labels.
    """

    def __init__(
        self,
        max_depth=3,
        number_of_trees=100,
        max_features=None,
        learning_rate=0.1,
        random_state=None,
    ):
        self.max_depth = max_depth
        self.number_of_trees = number_of_trees
        self.max_features = max_features
        self.learning_rate = learning_rate
        self.random_state = random_state

        self.trees = []
        self.initial_prediction = None
        self.classes_ = None

    def _sigmoid(self, x):
        """Apply the sigmoid function to convert log-odds to probabilities."""
        return 1 / (1 + np.exp(-x))

    def _compute_initial_prediction(self, y):
        """
        Compute the initial log-odds prediction from class priors.

        Returns log(pos / neg). Returns 0.0 if one class is absent.
        """
        pos = np.sum(y == 1)
        neg = np.sum(y == 0)
        if pos == 0 or neg == 0:
            return 0.0
        return np.log(pos / neg)

    def fit(self, X, y):
        """
        Fit the gradient boosting model.

        Parameters
        ----------
        X : array-like of shape (n_samples, n_features)
            Training features.
        y : array-like of shape (n_samples,)
            Binary target values.
        """
        if isinstance(X, pd.DataFrame):
            X = X.values
        if isinstance(y, pd.Series):
            y = y.values

        self.classes_ = np.unique(y)

        self.initial_prediction = self._compute_initial_prediction(y)
        F = np.full(len(y), self.initial_prediction)  # log-odds

        for tree_idx in range(self.number_of_trees):
            p = self._sigmoid(F)
            residuals = y - p  # negative gradient of logistic loss

            # Derive a unique seed per tree when random_state is set
            seed = None
            if self.random_state is not None:
                seed = self.random_state + tree_idx

            tree = DecisionTreeRegressor_custom(
                max_depth=self.max_depth,
                max_features=self.max_features,
                random_state=seed,
            )

            tree.fit(X, residuals)
            self.trees.append(tree)

            F += self.learning_rate * tree.predict(X)

    def add_tree(self, X, y):
        """
        Add a single additional boosting iteration to an already fitted model.

        Parameters
        ----------
        X : array-like of shape (n_samples, n_features)
            Training features.
        y : array-like of shape (n_samples,)
            Binary target values.

        Raises
        ------
        ValueError
            If the model has not been fitted yet.
        """
        if isinstance(X, pd.DataFrame):
            X = X.values
        if isinstance(y, pd.Series):
            y = y.values

        if self.initial_prediction is None:
            raise ValueError("Model is not fitted. Call fit() first.")

        F = self._get_raw_predictions(X)
        p = self._sigmoid(F)
        residuals = y - p

        seed = None
        if self.random_state is not None:
            seed = self.random_state + len(self.trees)

        tree = DecisionTreeRegressor_custom(
            max_depth=self.max_depth, max_features=self.max_features, random_state=seed
        )

        tree.fit(X, residuals)
        self.trees.append(tree)

    def _get_raw_predictions(self, X):
        """
        Compute raw log-odds predictions from the ensemble.

        Starts from the initial log-odds and adds the weighted contribution
        of each fitted tree.
        """
        if isinstance(X, pd.DataFrame):
            X = X.values

        F = np.full(len(X), self.initial_prediction)

        for tree in self.trees:
            F += self.learning_rate * tree.predict(X)

        return F

    def predict_proba(self, X):
        """
        Predict class probabilities.

        Returns
        -------
        proba : ndarray of shape (n_samples, 2)
            Column 0: probability of class 0.
            Column 1: probability of class 1.
        """
        F = self._get_raw_predictions(X)
        proba = self._sigmoid(F)
        return np.column_stack([1 - proba, proba])

    def predict(self, X):
        """
        Predict class labels.

        Returns
        -------
        y_pred : ndarray of shape (n_samples,)
            Predicted class label (0 or 1) based on the highest probability.
        """
        proba = self.predict_proba(X)
        return np.argmax(proba, axis=1)

In [27]:
gbdt = GBDTClassifier_custom(
    max_depth=3,
    number_of_trees=50,
    max_features="sqrt",
    learning_rate=0.1,
    random_state=21,
)

gbdt.fit(X_train_processed, y_train)
y_pred_proba = gbdt.predict_proba(X_valid_processed)[:, 1]
gini = 2 * roc_auc_score(y_valid, y_pred_proba) - 1
results = update_model_results(results, "GBDTClassifier_custom", "valid", gini)
gini

0.4608487691920444

## 6. LightGBM, Catboost, and XGBoost 

| Aspect | XGBoost | LightGBM | CatBoost |
|---|---|---|---|
| **Core Algorithm** | Gradient Boosting Decision Trees (GBDT) | GBDT with GOSS and EFB optimizations | Ordered Boosting + Symmetric Trees |
| **Implementation Language** | C++ | C++ | C++ / Python (core) |
| **Tree Growth Strategy** | **Level-wise:** The tree grows in a balanced manner — an entire level is filled before moving to the next | **Leaf-wise:** The leaf with the highest information gain (`max_delta_loss`) is split. Trees are deeper and more asymmetric | **Symmetric Trees (Oblivious):** At each level, all nodes use the same splitting condition. The tree is balanced and complete |
| **Growth Strategy Pros** | Good built-in regularization; lower risk of structural overfitting | Very fast training; better quality on complex dependencies; fewer nodes to achieve low loss | Very fast inference; small model footprint; CPU-efficient |
| **Growth Strategy Cons** | Slower training; higher memory usage (many redundant splits) | High risk of overfitting without proper constraints (`max_depth`, `min_data_in_leaf`) | May require more iterations for comparable quality; less flexible structure |
| **Handling Categorical Features** | **Manual encoding only** (One-Hot, Label). From v1.3: `enable_categorical=True`, essentially a wrapper | **Native, but requires care.** Algorithm sorts histograms by accumulated gradient. Must specify `categorical_feature` and convert to `int` | **Native and maximally safe.** Uses Ordered Target Encoding (no data leakage, with permutations). Gold standard for high-cardinality categories (e.g., user IDs) |
| **Handling Missing Values (NaN)** | **Sparsity-aware:** Automatically chooses the optimal direction (left/right child) for missing values during training | Handles natively. Treats NaN as a separate bin (histogram-based method) | Handles natively. In symmetric trees, missing values participate in splits in the standard way |
| **Key Innovation / "Killer Feature"** | **DART (Dropout Additive Regression Trees):** Dropout for trees to combat overfitting. <br> **Monotonic Constraints:** Business rules can be enforced (e.g., "as salary increases, risk score does not decrease") | **GOSS (Gradient-based One-Side Sampling):** Samples instances with small gradients (already well-predicted), keeping all instances with large gradients. <br> **EFB:** Bundles mutually exclusive features into one for data compression | **Ordered Boosting:** Solves the prediction shift problem by training on the "past" relative to a random permutation. <br> **CTR features:** On-the-fly categorical feature combinations |
| **Regularization (Overfitting Control)** | Richest set: `lambda` (L2), `alpha` (L1), `gamma` (min loss reduction), `min_child_weight`, `max_delta_step`, plus DART | `lambda_l1`, `lambda_l2`, `min_gain_to_split`, `min_data_in_leaf`, `path_smooth`, can constrain leaf sharpness | `l2_leaf_reg`, `random_strength` (noise to split boundaries), `depth`, `bagging_temperature` (Bayesian bagging) |
| **Training Speed** | Medium (Level-wise requires more computation) | **Very high** (Leaf-wise + GOSS provides a massive speedup). Fastest on CPU | Slower on CPU (due to symmetric trees and Ordered Boosting). Fast on GPU (perfectly suited for parallelism) |
| **Inference Speed (predict)** | High | Medium (deep asymmetric trees require more `if/else` checks) | **Very high (best in class).** Symmetric trees act as look-up tables; ideal for production |
| **Tuning (Ease of Hyperparameter Optimization)** | Many hyperparameters. Easy to find a "sweet spot" via cross-validation | Very sensitive to `num_leaves` and `min_data_in_leaf`. Easy to overfit; careful validation is essential | **The "lazy" option.** Works excellently on near-default parameters. Low sensitivity to hyperparameters |
| **Handling Class Imbalance** | Only via `scale_pos_weight` | `scale_pos_weight` or `is_unbalance` flag (dynamically adjusts weights during training) | `auto_class_weights` (Balanced/SqrtBalanced), plus weight control directly in the loss function |
| **GPU Support** | Excellent | Excellent | Superior (best multi-GPU scalability) |
| **NLP / Text Handling** | No built-in support | No built-in support | **Built-in:** CatBoost can transform text into embeddings via `text_features` (using BERT-like models or Bag-of-Words) |

---

### Summary of Special Modes and Features

#### 1. How Does CatBoost Handle "Categorical Features"?

- **The Problem:** Standard Target Encoding causes **data leakage (Target Leakage)** because the target mean for a category is calculated using the target value of the instance itself.
- **CatBoost's Solution (Ordered TS):**
    1.  **Random Permutation:** The data is randomly shuffled multiple times.
    2.  **Artificial "Past":** For each instance, the categorical feature is transformed into a numerical value (target mean) using **only the instances appearing before it** in the permutation.
    3.  **Prior Smoothing:** Uses the formula `(CountInClass + Prior) / (TotalCount + 1)`, blending the category's statistic with the global mean to combat noise.
    - **Result:** You obtain a powerful feature with **no risk of data leakage**, even for categories with a single representative.

#### 2. What is DART in XGBoost?

- **Mechanics:** When building each new iteration (new tree), DART randomly **drops** a portion of already-built trees from the ensemble with probability `rate_drop`.
- **Why is this needed?** In classical GBDT, the first few trees contribute the bulk of the prediction. Later trees often merely "fit to the noise" left over by the early trees, leading to overfitting. DART forces new trees to learn to compensate for the absence of major components, making the ensemble more robust and better at generalization.

#### 3. Why One Model Might Outperform the Others

Given task (binary classification, time-based split, manual categorical preprocessing):

1.  **LightGBM Wins If:** You have a large dataset with complex non-linear interactions. Its leaf-wise strategy builds more precise decision boundaries without wasting resources on balancing the tree. GOSS allows it to ignore already well-predicted "easy" examples.

2.  **XGBoost DART Wins If:** The model is prone to severe overfitting (e.g., Train AUC = 0.99, Validation AUC = 0.85). The random dropping of trees acts as a strong stabilizer.



| Аспект | XGBoost | LightGBM | CatBoost |
|---|---|---|---|
| **Базовый алгоритм** | Gradient Boosting Decision Trees (GBDT) | GBDT с оптимизациями GOSS и EFB | Ordered Boosting + Symmetric Trees |
| **Язык разработки** | C++ | C++ | C++ / Python (ядро) |
| **Стратегия роста деревьев** | **Level-wise (поуровневый):** дерево растет сбалансированно, сначала заполняется весь уровень, затем следующий | **Leaf-wise (по листьям):** выбирается лист с максимальным приростом информации (`max_delta_loss`). Деревья глубже и асимметричнее | **Symmetric Trees (симметричные, Oblivious):** на каждом уровне все узлы используют одинаковое условие разделения. Дерево — сбалансированное и полное |
| **Плюсы стратегии роста** | Хорошая регуляризация "из коробки", меньше риск переобучения на уровне структуры | Очень быстрое обучение, лучшее качество на сложных зависимостях, меньше узлов для достижения низкого лосса | Очень быстрое предсказание (инференс), модель занимает мало памяти, эффективно на CPU |
| **Минусы стратегии роста** | Медленнее обучается, большее потребление памяти (много лишних разбиений) | Высокий риск переобучения без контроля (`max_depth`, `min_data_in_leaf`) | Может требовать больше итераций для качества, менее гибкая структура |
| **Работа с категориальными признаками** | **Только ручное кодирование** (One-Hot, Label). Начиная с v1.3 — поддержка `enable_categorical=True`, но это просто обертка | **Нативная, но требует аккуратности.** Алгоритм сортирует гистограммы по накопленному градиенту. Необходимо указывать `categorical_feature` и преобразовывать в `int` | **Нативная и максимально безопасная.** Использует Ordered Target Encoding (без утечки данных, с пермутациями). Золотой стандарт для категорий высокой кардинальности (например, ID пользователя) |
| **Обработка пропущенных значений (NaN)** | **Sparsity-aware:** автоматически выбирает оптимальное направление (левый/правый потомок) для пропусков при обучении | Обрабатывает нативно. Считает, что NaN — это отдельный "бин" (гистограммный метод) | Обрабатывает нативно. В симметричных деревьях пропуски участвуют в разделении стандартным образом |
| **Ключевая "фича" (Инновация)** | **DART (Dropout Additive Regression Trees):** Дропаут для деревьев для борьбы с переобучением. <br> **Monotonic Constraints:** Можно задать бизнес-правила (например, "с ростом зарплаты скоринг не падает") | **GOSS (Gradient-based One-Side Sampling):** Сэмплирует только объекты с маленькими градиентами (уже хорошо обученные), сохраняя все с большими. <br> **EFB:** Упаковывает взаимоисключающие признаки в один для сжатия данных | **Ordered Boosting:** Решает проблему сдвига предсказаний (Prediction Shift), обучаясь на "прошлом" относительно случайной перестановки. <br> **CTR features:** Комбинации категорий на лету |
| **Регуляризация (борьба с переобучением)** | Самая богатая: `lambda` (L2), `alpha` (L1), `gamma` (min loss reduction), `min_child_weight`, `max_delta_step`, а также DART | `lambda_l1`, `lambda_l2`, `min_gain_to_split`, `min_data_in_leaf`, `path_smooth`, можно ограничить резкость листьев | `l2_leaf_reg`, `random_strength` (зашумление границ сплитов), `depth`, `bagging_temperature` (байесовский бэггинг) |
| **Скорость обучения** | Средняя (Level-wise требует больше вычислений) | **Очень высокая** (Leaf-wise + GOSS дают огромный буст). Быстрее всех на CPU | Низкая на CPU (из-за симметричных деревьев и Ordered Boosting). Высокая на GPU (идеально ложится на параллелизм) |
| **Скорость инференса (predict)** | Высокая | Средняя (глубокие асимметричные деревья требуют больше проверок `if/else`) | **Очень высокая (лучшая).** Симметричные деревья — это просто таблицы (look-up tables), идеально для продакшена |
| **Тюнинг (Сложность настройки)** | Много гиперпараметров. Легко найти "золотую середину" на кросс-валидации | Очень чувствителен к `num_leaves` и `min_data_in_leaf`. Легко уйти в переобучение, нужно тщательно валидировать | **Самый "ленивый".** Работает отлично почти на дефолтных параметрах. Мало чувствителен к гиперпараметрам |
| **Работа с дисбалансом классов** | Только через `scale_pos_weight` | `scale_pos_weight` или флаг `is_unbalance` (автоматически меняет веса по ходу обучения) | `auto_class_weights` (Balanced/SqrtBalanced), а также контроль веса прямо в функции потерь |
| **GPU поддержка** | Отличная | Отличная | Превосходная (лучшая масштабируемость на мульти-GPU) |
| **Работа с текстом (NLP)** | Нет встроенной поддержки | Нет встроенной поддержки | **Встроенная:** CatBoost умеет превращать текст в эмбеддинги через `text_features` (на основе BERT-like моделей или BoW) |

---

### Сводка по особым режимам и фичам

#### 1. Как работает "categorical feature" в CatBoost?
*   **Проблема:** Обычный Target Encoding приводит к утечке данных (Target Leakage), так как среднее значение таргета для группы считается с использованием самой целевой переменной объекта.
*   **Решение CatBoost (Ordered TS):**
    1.  **Случайная перестановка:** Данные многократно случайно перемешиваются.
    2.  **Искусственное "прошлое":** Для каждого объекта категориальный признак преобразуется в числовое значение (среднее по таргету), используя *только те объекты, которые идут в перестановке перед ним*.
    3.  **Приоризация:** Использует формулу `(CountInClass + Prior) / (TotalCount + 1)`, смешивая статистику с глобальным средним для борьбы с шумом.
    *   **Итог:** Вы получаете мощный признак без риска утечки данных, даже для категорий с 1 представителем.

#### 2. Что такое DART в XGBoost?
*   **Механика:** При построении каждой новой итерации (нового дерева) DART с вероятностью `rate_drop` "выкидывает" из ансамбля часть уже построенных деревьев.
*   **Зачем это нужно?** В классическом GBDT первые несколько деревьев вносят основной вклад. Новые деревья часто лишь занимаются "подгонкой" шума, оставленного первыми деревьями, что ведет к переобучению. DROP заставляет новые деревья учиться компенсировать отсутствие крупных игроков, делая ансамбль более устойчивым и обобщающим.


### 6.1. Light Gradient Boosting Machine (Microsoft)

In [28]:
X = X_copy.copy()
y = y.copy()
X = X.drop(columns="RefId")

In [29]:
obj_cols = X.select_dtypes(include=["object"]).columns.drop("PurchDate")
X[obj_cols] = X[obj_cols].astype("category")

In [30]:
X_train, X_valid, X_test, y_train, y_valid, y_test = time_ordered_split_3_new(
    X,
    y,
    date_field="PurchDate",
    train_size=1 / 3,
    valid_size=1 / 3,
    test_size=1 / 3,
)
X_train = X_train.drop(columns="PurchDate")
X_valid = X_valid.drop(columns="PurchDate")
X_test = X_test.drop(columns="PurchDate")

In [31]:
model_lgb = lgb.LGBMClassifier(
    # is_unbalance=True,
    n_estimators=2000,
    learning_rate=0.01,
    num_leaves=63,
    n_jobs=-1,
    random_state=21,
)

model_lgb.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    callbacks=[lgb.early_stopping(stopping_rounds=50)],
)

y_pred_proba = model_lgb.predict_proba(X_valid)[:, 1]
gini = 2 * roc_auc_score(y_valid, y_pred_proba) - 1
results = update_model_results(results, "LGBMClassifier", "valid", gini)
gini

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 2607, number of negative: 20452
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002072 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4026
[LightGBM] [Info] Number of data points in the train set: 23059, number of used features: 27
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.113058 -> initscore=-2.059881
[LightGBM] [Info] Start training from score -2.059881
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[227]	valid_0's binary_logloss: 0.330966


0.4686040608041391

### 6.2. Categorical Boosting (Яндекс)

In [32]:
X = X_copy.copy()
y = y.copy()
X = X.drop(columns="RefId")

In [33]:
cat_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
X[cat_features] = X[cat_features].astype(str).replace("nan", "NaN")

X_train, X_valid, X_test, y_train, y_valid, y_test = time_ordered_split_3_new(
    X,
    y,
    date_field="PurchDate",
    train_size=1 / 3,
    valid_size=1 / 3,
    test_size=1 / 3,
)
X_train = X_train.drop(columns="PurchDate")
X_valid = X_valid.drop(columns="PurchDate")
X_test = X_test.drop(columns="PurchDate")

cat_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

In [34]:
model_cb = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    auto_class_weights="Balanced",
    random_seed=21,
    verbose=100,
)

model_cb.fit(
    X_train,
    y_train,
    cat_features=cat_features,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=50,
)

y_pred_proba = model_cb.predict_proba(X_valid)[:, 1]
gini = 2 * roc_auc_score(y_valid, y_pred_proba) - 1
results = update_model_results(results, "CatBoostClassifier", "valid", gini)
gini

0:	learn: 0.6814203	test: 0.6814136	best: 0.6814136 (0)	total: 85.5ms	remaining: 1m 25s
100:	learn: 0.5542126	test: 0.5910177	best: 0.5907153 (97)	total: 2.22s	remaining: 19.8s
200:	learn: 0.5360212	test: 0.5881889	best: 0.5881791 (198)	total: 4.33s	remaining: 17.2s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.5881790712
bestIteration = 198

Shrink model to first 199 iterations.


0.4929216952977691

### 6.3. eXtreme Gradient Boosting (Open Source)

In [35]:
X = X_copy.copy()
y = y.copy()
X = X.drop(columns="RefId")

In [36]:
obj_cols = X.select_dtypes(include=["object"]).columns.drop("PurchDate")
X[obj_cols] = X[obj_cols].astype("category")

In [37]:
X_train, X_valid, X_test, y_train, y_valid, y_test = time_ordered_split_3_new(
    X,
    y,
    date_field="PurchDate",
    train_size=1 / 3,
    valid_size=1 / 3,
    test_size=1 / 3,
)
X_train = X_train.drop(columns="PurchDate")
X_valid = X_valid.drop(columns="PurchDate")
X_test = X_test.drop(columns="PurchDate")

In [38]:
model_xgb = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    enable_categorical=True,
    tree_method="hist",
    random_state=21,
    early_stopping_rounds=50,
    # scale_pos_weight=(len(y_train) - y_train.sum()) / y_train.sum() # Balanced
)

model_xgb.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=100)

y_pred_proba = model_xgb.predict_proba(X_valid)[:, 1]
gini = 2 * roc_auc_score(y_valid, y_pred_proba) - 1
results = update_model_results(results, "XGBClassifier", "valid", gini)
gini

[0]	validation_0-logloss:0.38456
[100]	validation_0-logloss:0.33015
[109]	validation_0-logloss:0.33051


0.47423057390084034

## 7. The best model

In [39]:
results

,model,data_type,gini
0,DecisionTreeClassifier_custom,valid,0.420428
1,DecisionTreeClassifier,valid,0.420630
2,RandomForestClassifier_custom,valid,0.469876
3,GBDTClassifier_custom,valid,0.460849
4,LGBMClassifier,valid,0.468604
5,CatBoostClassifier,valid,0.492922
6,XGBClassifier,valid,0.474231


Why CatBoost won:

1. CatBoost's victory is driven by the perfect match between the algorithm's architecture and the specifics of the data — severe class imbalance and an abundance of categorical features.
2. The key advantage is **Ordered Target Encoding**, which encodes categories using only past instances in a random permutation, completely eliminating target leakage.
3. Unlike LightGBM, whose primitive target encoding generates noise under class imbalance, CatBoost delivers a statistically honest signal.
4. XGBoost, on the other hand, cannot handle categories natively at all, requiring manual encoding, which is almost always inferior to a built-in solution.
5. The second factor is **symmetric trees**, which act as a built-in regularizer, preventing the model from overfitting to outliers of the rare class.
6. Ordered Boosting further stabilizes training on imbalanced data, providing unbiased gradient estimates where standard methods fail.
7. The custom implementations (DecisionTree, GBDT) lost predictably, as they lack both proper categorical handling and mechanisms to cope with class imbalance.
8. CatBoost did not win through hyperparameter tuning, but by virtue of its architecture, which inherently provides an advantage in the "many categories + few positive examples" scenario.


In [40]:
results = pd.DataFrame()

In [41]:
X = X_copy.copy()
y = y.copy()
X = X.drop(columns="RefId")

cat_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
X[cat_features] = X[cat_features].astype(str).replace("nan", "NaN")

X_train, X_valid, X_test, y_train, y_valid, y_test = time_ordered_split_3_new(
    X,
    y,
    date_field="PurchDate",
    train_size=1 / 3,
    valid_size=1 / 3,
    test_size=1 / 3,
)
X_train = X_train.drop(columns="PurchDate")
X_valid = X_valid.drop(columns="PurchDate")
X_test = X_test.drop(columns="PurchDate")

In [42]:
y_pred_proba = model_cb.predict_proba(X_train)[:, 1]
gini = 2 * roc_auc_score(y_train, y_pred_proba) - 1
results = update_model_results(results, "CatBoostClassifier", "train", gini)
gini

0.5866750337651021

In [43]:
y_pred_proba = model_cb.predict_proba(X_valid)[:, 1]
gini = 2 * roc_auc_score(y_valid, y_pred_proba) - 1
results = update_model_results(results, "CatBoostClassifier", "valid", gini)
gini

0.4929216952977691

In [44]:
y_pred_proba = model_cb.predict_proba(X_test)[:, 1]
gini = 2 * roc_auc_score(y_test, y_pred_proba) - 1
results = update_model_results(results, "CatBoostClassifier", "test", gini)
gini

0.48567397841493776

In [45]:
results

,model,data_type,gini
0,CatBoostClassifier,train,0.586675
1,CatBoostClassifier,valid,0.492922
2,CatBoostClassifier,test,0.485674


The model demonstrates stability and the ability to correctly interpret test data.

## 8.  ExtraTreesClassifier

In [46]:
class ExtraTreesClassifier_custom(DecisionTreeClassifier):
    """
    An Extremely Randomized Trees classifier.

    Extends DecisionTreeClassifier_custom by randomizing both the feature
    subset and the split threshold at each node. Instead of searching for
    the optimal threshold among all unique feature values, it draws a
    single random threshold from a uniform distribution between the
    feature's minimum and maximum values for each candidate feature.

    This approach trades the exhaustive threshold search of a standard
    decision tree for additional randomness, which typically reduces
    variance and computational cost at the expense of slightly higher bias
    per tree. It is designed to be used as a base estimator in ensemble
    methods like Random Forest or as a standalone classifier.

    Parameters
    ----------
    max_depth : int, default=7
        Maximum depth of the tree. Passed to the parent
        DecisionTreeClassifier_custom.
    max_features : int, str, or None, default=None
        Number of features to randomly sample at each split.
        - If None, all features are considered.
        - If int, that many features are randomly chosen.
        Note: Unlike the parent class, "sqrt" string is not handled
        automatically; if needed, the caller should compute
        int(sqrt(total_features)) and pass it explicitly, or the
        behavior is determined by the parent class initialization.
    random_state : int or None, default=None
        Seed for the random number generator. A dedicated
        ``numpy.random.RandomState`` instance is created to ensure
        all random operations (feature selection and threshold
        generation) are reproducible.

    Attributes
    ----------
    rng : numpy.random.RandomState
        A seeded random number generator instance used for all
        stochastic operations in the tree (feature subsampling and
        random threshold generation).
    max_features : int, str, or None
        Stored value of the max_features parameter.
    random_state : int or None
        Stored seed value.

    Notes
    -----
    Key differences from DecisionTreeClassifier_custom._best_split:

    - **Feature selection**: Randomly samples a subset of features
      using the dedicated ``rng`` generator instead of the global
      ``np.random`` seed.
    - **Threshold selection**: For each sampled feature, generates
      a single random threshold from ``Uniform(f_min, f_max)``
      instead of testing all unique values. Constant features
      (where f_min == f_max) are skipped.
    - **Reproducibility**: All randomness is controlled by the
      ``self.rng`` RandomState instance, isolating the tree's
      stochastic behavior from the global numpy random state and
      from other trees in an ensemble.
    """

    def __init__(self, max_depth=7, max_features=None, random_state=None):
        super().__init__(max_depth=max_depth)
        self.max_features = max_features
        self.random_state = random_state
        self.rng = np.random.RandomState(random_state)

    def _best_split(self, X, y, total_features):
        """
        Find the best split using the Extra-Trees randomized algorithm.

        For each randomly selected feature, a single random threshold
        is drawn from a uniform distribution over the feature's range
        [f_min, f_max]. The split yielding the highest information
        gain (Gini reduction) among all sampled features is returned.
        Constant features are skipped.

        Parameters
        ----------
        X : np.ndarray of shape (n_samples, n_features)
            Feature matrix for the current node.
        y : np.ndarray of shape (n_samples,)
            Target labels for the current node.
        total_features : int
            Total number of features available in X.

        Returns
        -------
        split_idx : int or None
            Index of the best feature to split on, or None if no
            valid split exists (e.g., all features are constant).
        split_thresh : float or None
            Randomly generated threshold for the best split, or
            None if no valid split exists.
        """
        best_gain = -1
        split_idx, split_thresh = None, None

        parent_gini = self._gini(y)
        
        n_features_to_sample = self.max_features if self.max_features else total_features
        
        feature_indices = self.rng.choice(total_features, n_features_to_sample, replace=False)

        for feat_idx in feature_indices:
            X_column = X[:, feat_idx]

            f_min, f_max = np.min(X_column), np.max(X_column)
            
            if f_min == f_max:
                continue

            random_thresh = self.rng.uniform(f_min, f_max)

            gain = self._information_gain(y, X_column, random_thresh, parent_gini)
            
            if gain > best_gain:
                best_gain = gain
                split_idx = feat_idx
                split_thresh = random_thresh

        return split_idx, split_thresh

In [47]:
etc = ExtraTreesClassifier_custom(max_depth=5, max_features=int(np.sqrt(10)), random_state=21)
etc.fit(X_train_processed, y_train)
y_pred_proba = etc.predict_proba(X_valid_processed)[:, 1]
gini = 2 * roc_auc_score(y_valid, y_pred_proba) - 1
results = update_model_results(results, "ExtraTreesClassifier_custom", "valid", gini)
gini

0.34812823422517725

In [48]:
results

,model,data_type,gini
0,CatBoostClassifier,train,0.586675
1,CatBoostClassifier,valid,0.492922
2,CatBoostClassifier,test,0.485674
3,ExtraTreesClassifier_custom,valid,0.348128
